## Weighted quadrature

Standard Gauss quadrature uses `degree + 1` points per element, all basis functions
sharing the same weight at a given point. **Weighted quadrature (WQ)** instead uses a
*reduced*, globally-chosen set of points, with each basis function carrying its *own*
weight at each point (precomputed once by solving a small least-squares problem per
basis function). Fewer points to evaluate the geometry/material at is the whole point --
for a tensor-product 2D/3D IGA problem the savings compound across directions, which is
the main motivation for matrix-free IGA solvers.

`WeightedQuadrature1D` (`src/yeti_iga/future/include/WeightedQuadrature1D.hpp`) is a
direct C++ port of `yeti_iga.pymfiga`'s `weighted_quadrature.py` -- both variants,
**method "1" and method "2"** (`build()`'s second argument, `pymfiga`'s own naming).
`pymfiga`'s own validated regression benchmarks (`benchs/pymfiga/`) use
`QUADCLASS="wq"`, `QUADTYPE="2"`, so method "2" is the default here too -- but both are
built and compared side-by-side throughout this notebook, not just method "2" alone.
Reference: Calabro, Sangalli, Tani (2017), *"Fast formation of isogeometric Galerkin
matrices by weighted quadrature"*.

**Scope of what's below**: only the 1D quadrature rule itself (points, basis values,
weights) is implemented so far. The matrix-free Kronecker-product application that
*uses* this rule to compute a stiffness/mass action without ever assembling a matrix
(`pymfiga`'s `MatrixFree.apply`) is a separate, not-yet-ported piece -- see *Next steps*
at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import BSpline, WeightedQuadrature1D, IGABasis1D

## Fewer points than Gauss

A degree-3 B-spline over 10 elements: standard Gauss needs `4` points per element,
`degree + 1`, everywhere. WQ needs a reduced, globally-computed set instead -- built here
for **both** `pymfiga` methods (`wq1` = method "1", `wq2` = method "2") so the two can be
compared directly.

In [ ]:
degree = 3
nbel = 10
interior_knots = np.linspace(0.0, 1.0, nbel + 1)[1:-1]
kv = np.array([0.0] * (degree + 1) + list(interior_knots) + [1.0] * (degree + 1))

bsp = BSpline(degree, kv)
nbctrlpts = len(kv) - degree - 1

wq1 = WeightedQuadrature1D.build(bsp, "1")
wq2 = WeightedQuadrature1D.build(bsp, "2")
gauss = IGABasis1D.build(bsp, degree + 1, 1)
gauss_pts = [u for sg in gauss.gauss_spans for u in sg.u_param]

print(f'degree {degree}, {nbel} elements, {nbctrlpts} control points')
print(f'Gauss points:          {len(gauss_pts)}  ({degree + 1} per element)')
print(f'WQ points, method "1": {len(wq1.quadpts)}')
print(f'WQ points, method "2": {len(wq2.quadpts)}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.plot(gauss_pts, np.full(len(gauss_pts), 2), 'o', color='C0', label=f'Gauss ({len(gauss_pts)} pts)')
ax.plot(wq1.quadpts, np.full(len(wq1.quadpts), 1), 's', color='C1', label=f'WQ method "1" ({len(wq1.quadpts)} pts)')
ax.plot(wq2.quadpts, np.full(len(wq2.quadpts), 0), 's', color='C2', label=f'WQ method "2" ({len(wq2.quadpts)} pts)')
for k in np.unique(kv):
    ax.axvline(k, color='gray', lw=0.5, zorder=0)
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(['WQ "2"', 'WQ "1"', 'Gauss'])
ax.set_xlabel('u'); ax.set_ylim(-0.5, 2.5)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.45), ncol=3)
plt.tight_layout(); plt.show()

The gap widens with more elements for both methods (savings grow with degree too, and
for a 2D/3D tensor-product problem the *per-direction* savings multiply -- 23% fewer
points per direction is already ~40% fewer for a 2D problem, ~54% for 3D). Method "1"
places fewer points than method "2" at every element count -- the tradeoff is a
lower-degree target space (see the table further down):

In [ ]:
nbel_list = [4, 8, 16, 32, 64]
n_gauss_list, n_wq1_list, n_wq2_list = [], [], []
for n in nbel_list:
    kv_n = np.array([0.0] * (degree + 1) + list(np.linspace(0, 1, n + 1)[1:-1]) + [1.0] * (degree + 1))
    bsp_n = BSpline(degree, kv_n)
    n_gauss_list.append(n * (degree + 1))
    n_wq1_list.append(len(WeightedQuadrature1D.build(bsp_n, "1").quadpts))
    n_wq2_list.append(len(WeightedQuadrature1D.build(bsp_n, "2").quadpts))

fig, ax = plt.subplots()
ax.plot(nbel_list, n_gauss_list, 'o-', label='Gauss')
ax.plot(nbel_list, n_wq1_list, 's-', label='WQ method "1"')
ax.plot(nbel_list, n_wq2_list, '^-', label='WQ method "2"')
ax.set_xlabel('number of elements'); ax.set_ylabel('number of quadrature points')
ax.legend(); plt.show()

## What `WeightedQuadrature1D` actually holds

- `B0`, `B1`: basis values / first derivatives at the WQ points, shape `(n_wq, n_ctrlpts)`
  -- same role as `IGABasis1D`'s per-span `N`/`dN`, just evaluated at every WQ point at
  once, globally, rather than per span.
- `W00`, `W01`, `W10`, `W11`: the WQ weights, shape `(n_ctrlpts, n_wq)` -- one weight
  *per (basis function, point) pair*, not one shared weight per point like Gauss. For
  **method "2"**, `W00==W01` and `W10==W11` (only 2 distinct least-squares solves per
  basis function); for **method "1"** all four differ (4 distinct solves) -- checked for
  both below.

| | points/interior element | points/boundary element | target space | distinct weights |
|---|---|---|---|---|
| method "1" | `2 + s = 3` | `r = degree + 2` | degree `p-1`, same interior knots | 4 |
| method "2" | `2 + s = 4` | `r = degree + 3` | degree `p`, +1 interior knot multiplicity | 2 |

In [ ]:
for label, wq in [('method "1"', wq1), ('method "2"', wq2)]:
    print(f'{label}: B0 shape {wq.B0.shape}, nnz {wq.B0.nnz}  |  '
          f'W00 shape {wq.W00.shape}, nnz {wq.W00.nnz}  |  '
          f'W00==W01: {np.allclose(wq.W00.toarray(), wq.W01.toarray())}  |  '
          f'W10==W11: {np.allclose(wq.W10.toarray(), wq.W11.toarray())}')

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, (label, wq) in enumerate([('method "1"', wq1), ('method "2"', wq2)]):
    axes[row, 0].spy(wq.B0.toarray(), markersize=3)
    axes[row, 0].set_title(f'{label}: B0 (n_wq x n_ctrlpts)')
    axes[row, 1].spy(wq.W00.toarray(), markersize=3)
    axes[row, 1].set_title(f'{label}: W00 (n_ctrlpts x n_wq)')
plt.tight_layout(); plt.show()

## Cross-check against the `pymfiga` Python reference

`WeightedQuadrature1D` is a direct port -- validated here the same way as
`tests/future/test_weighted_quadrature.py`: comparing every quantity (points, basis
values, and all four weight matrices) against `pymfiga`'s own (already-validated)
Python implementation, for **both** methods.

In [ ]:
from yeti_iga.pymfiga.common.numerics.quadrature_rules.weighted_quadrature import WeightedQuadrature

for quadtype, wq in [("1", wq1), ("2", wq2)]:
    wq_py = WeightedQuadrature(degree, kv, quadtype=quadtype)
    wq_py.export_quadrature_rules()
    print(f'method "{quadtype}":')
    print('  quadrature points match:', np.allclose(wq.quadpts, wq_py.quadpts, atol=1e-10))
    print('  B0 match:', np.allclose(wq.B0.toarray(), wq_py.basis[0].toarray(), atol=1e-8))
    print('  B1 match:', np.allclose(wq.B1.toarray(), wq_py.basis[1].toarray(), atol=1e-8))
    print('  W00 match:', np.allclose(wq.W00.toarray(), wq_py.weights[0].toarray(), atol=1e-6))
    print('  W01 match:', np.allclose(wq.W01.toarray(), wq_py.weights[1].toarray(), atol=1e-6))
    print('  W10 match:', np.allclose(wq.W10.toarray(), wq_py.weights[2].toarray(), atol=1e-6))
    print('  W11 match:', np.allclose(wq.W11.toarray(), wq_py.weights[3].toarray(), atol=1e-6))

## A concrete payoff: exact mass and stiffness matrices, fewer points

WQ's weights are constructed so that `W @ B` reproduces certain Galerkin-type integrals
exactly (not just approximately) for a spline space of this degree -- checked here for
**both methods** directly against the same 1D mass and stiffness matrices `future`'s own
Gauss-based `IGABasis1D` would give (with enough Gauss points, `degree + 1`, to integrate
the `2*degree`-degree product exactly).

In [ ]:
def gauss_mass_stiffness(bsp, basis, nbctrlpts, degree):
    M = np.zeros((nbctrlpts, nbctrlpts))
    K = np.zeros((nbctrlpts, nbctrlpts))
    for span, idx in basis.span_indices.items():
        sg = basis.gauss_spans[idx]
        for q in range(len(sg.u_param)):
            N, dN, w = sg.N[q], sg.dN[q], sg.weight[q]
            for a in range(degree + 1):
                for b in range(degree + 1):
                    M[span - degree + a, span - degree + b] += N[a] * N[b] * w
                    K[span - degree + a, span - degree + b] += dN[a] * dN[b] * w
    return M, K


M_gauss, K_gauss = gauss_mass_stiffness(bsp, gauss, nbctrlpts, degree)

for label, wq in [('method "1"', wq1), ('method "2"', wq2)]:
    B0, B1 = wq.B0.toarray(), wq.B1.toarray()
    W00, W10 = wq.W00.toarray(), wq.W10.toarray()
    M_wq = W00 @ B0
    K_wq = W10 @ B1
    err_M = np.linalg.norm(M_wq - M_gauss) / np.linalg.norm(M_gauss)
    err_K = np.linalg.norm(K_wq - K_gauss) / np.linalg.norm(K_gauss)
    print(f'{label} ({len(wq.quadpts)} points): '
          f'mass rel. error {err_M:.2e}, stiffness rel. error {err_K:.2e}')

Method "2" reproduces both matrices to machine precision, with fewer points than Gauss.
Method "1" is exact for the mass matrix too, but **not** for the stiffness matrix once
`degree >= 3` -- a persistent ~1e-3 relative error that does not shrink under mesh
refinement (try changing `degree`/`nbel` above). This is a genuine limitation of method
"1"'s lower-degree (`p-1`) target space, not a general WQ artifact -- and it is exactly
why `pymfiga`'s own validated regression benchmarks only exercise method "2".

Either way, these matrices were built here by forming `W @ B` -- i.e. *assembling* the
matrix. That is not yet the point of weighted quadrature: the actual payoff is a
matrix-free matrix-*vector* product (`W @ (B @ v)`, or the reverse for the transpose)
that never forms `W @ B` at all -- see *Next steps*.

## Next steps

- **Matrix-free Kronecker product** (`pymfiga`'s `MatrixFree.apply`, sum-factorization):
  apply a tensor-product operator to a vector via mode-wise contractions, without ever
  forming the full (Kronecker-product) matrix. Not yet ported.
- **Matrix-free elasticity stiffness action, curved geometry included**: combine the
  above with `PlaneStress`/`PlaneStrain`'s existing Lame-parameter physics, precomputing
  the pulled-back material tensor at each (few) WQ point and looping only over those
  points for the geometry-dependent part -- `pymfiga`'s `iga/single_model/mechanical.py`
  is the validated reference algorithm to port. Not yet implemented; the acceptance test
  will be comparing its matrix-vector product against `PatchIntegrator.integrateStiffness()
  @ v` on both a flat and a curved (NURBS) patch.